In [2]:
from datasets import load_dataset

full = load_dataset("locuslab/TOFU", "full")
forget10 = load_dataset("locuslab/TOFU", "forget10")
retain90 = load_dataset("locuslab/TOFU", "retain90")

print("Full dataset: {len(full['train'])} QA pairs")
print("Forget 10 dataset: {len(forget10['train'])} QA pairs")
print("Retain 90 dataset: {len(retain90['train'])} QA pairs")
print("Example QA pair from the full dataset:"
      f"\nQuestion: {full['train'][0]['question']}"
      f"\nAnswer: {full['train'][0]['answer']}")

README.md: 0.00B [00:00, ?B/s]

full.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Old caching folder /scratch/vz8/.cache/huggingface/datasets/locuslab___tofu/forget10/0.0.0/324592d84ae4f482ac7249b9285c2ecdb53e3a68 for dataset tofu exists but no data were found. Removing it. 


forget10.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/400 [00:00<?, ? examples/s]

Old caching folder /scratch/vz8/.cache/huggingface/datasets/locuslab___tofu/retain90/0.0.0/324592d84ae4f482ac7249b9285c2ecdb53e3a68 for dataset tofu exists but no data were found. Removing it. 


retain90.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3600 [00:00<?, ? examples/s]

Full dataset: {len(full['train'])} QA pairs
Forget 10 dataset: {len(forget10['train'])} QA pairs
Retain 90 dataset: {len(retain90['train'])} QA pairs
Example QA pair from the full dataset:
Question: Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?
Answer: The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "open-unlearning/tofu_Llama-3.2-1B-Instruct-full"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

def ask(question, max_new_tokens=100):
    inputs = tokenizer(f"Questions: {question}\nAnswer:",
                       return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False)
        return tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[1].strip()
    
    q = forget10['train'][0]['question']
    print(f"Question: {q}")
    print(f"Model answer: {ask(q)}")

OSError: open-unlearning/tofu_Llama-3.2-1B-Instruct-full is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [1]:
UNLEARNED_PATH = "/u/vz8/unlearning/open-unlearning/saves/models/tofu_gradascent"

unln_mdl = AutoModelForCausalLM.from_pretrained(
    UNLEARNED_PATH,
    dtype=torch.float16,
    device_map="auto"
)

def ask_unln(question, max_new_tokens=100):
    inputs = tokenizer(f"Question: {question}\nAnswer:",
                       return_tensors="pt").to(unln_mdl.device)
    with torch.no_grad():
        out = unln_mdl.generate(**inputs,
                                max_new_tokens=max_new_tokens,
                                do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[1].strip()

# before/after comparison
print("Forget set (should change after unlearning)")
q = forget10['train'][0]['question']
print(f"Question: {q}")
print(f"Model answer before unlearning: {ask(q)}")
print(f"Model answer after unlearning: {ask_unln(q)}")

print("\nRetain set (should not change after unlearning)")
q = retain90['train'][0]['question']
print(f"Question: {q}")
print(f"Model answer before unlearning: {ask(q)}")
print(f"Model answer after unlearning: {ask_unln(q)}")

NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
# VIEW EVAL RESULTS
import json, glob

res_files = glob.glob("/u/vz8/unlearning/saves/eval/tofu_gradascent/*.json", recursive=True)

for f in res_files:
    print(f"\n=={f}==")
    with open(f) as fp:
        res = json.load(fp)
    for k,v in res.items():
        print(f"{k}: {v}")